# Qwen2.5-3B Inference Server — Kaggle + FastAPI + ngrok

This notebook loads `Qwen2.5-3B-Instruct`, serves it through a FastAPI `/generate` endpoint, and exposes that endpoint publicly via ngrok so you can call it from your local machine.

**Before running, set up the notebook environment:**
1. Settings (right panel) → **Internet** → **On**
2. Settings → **Accelerator** → **GPU T4 x2** (or P100)
3. In the **Config** cell below, fill in your `NGROK_AUTH_TOKEN` (from https://dashboard.ngrok.com/get-started/your-authtoken) and choose your own `APP_API_KEY`
4. Run all cells in order. The last cell will block (this is expected) — it IS the running server.
5. Copy the printed `Public URL` into your local client script to send requests.

## 1. Install dependencies

In [1]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.7 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

## 2. Config — edit these values before running

In [2]:
# ====== EDIT THESE BEFORE RUNNING ======
NGROK_AUTH_TOKEN = "39yMUnnUJ4IK8wmbAuLvv84UTQY_4FsKCUbEmyzxznqodBwQk"      # from https://dashboard.ngrok.com/get-started/your-authtoken
APP_API_KEY = "qwen-server-prod-2026-7f9a4c2d8e1b5f6a"   # your own secret, used to protect the API endpoint (not the ngrok token)
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
PORT = 8000
# ========================================

## 3. Imports

In [3]:
import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from pyngrok import ngrok

## 4. Load the model

In [4]:
print("Loading model... this can take a minute.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16, 
    device_map="auto",
)
model.eval()
print("Model loaded.")

Loading model... this can take a minute.


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


## 5. Define the FastAPI app

In [5]:
app = FastAPI(title="Qwen2.5-3B Inference Server")

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 256
    temperature: float = 0.7

class GenerateResponse(BaseModel):
    response: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest, x_api_key: str = Header(default="")):
    if x_api_key != APP_API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    messages = [{"role": "user", "content": req.prompt}]

    # Step 1: get the formatted string from chat template
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,          # returns a plain string, not tensors
    )

    # Step 2: tokenize the string normally — guaranteed to return a plain tensor
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    # Step 3: generate
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=inputs["attention_mask"],
            max_new_tokens=req.max_new_tokens,
            temperature=req.temperature if req.temperature > 0 else 1.0,
            do_sample=req.temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output_ids[0][input_ids.shape[-1]:]
    text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return GenerateResponse(response=text)

## 6. Start the ngrok tunnel and run the server\n\nThis cell **blocks** — that is expected, it is the running server. Copy the printed `Public URL` and use it from your local client.

In [6]:
import asyncio
import threading
import time

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)

def run_server():
    asyncio.set_event_loop(asyncio.new_event_loop())
    server.run()

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # give uvicorn a moment to bind the port

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(addr=PORT, proto="http")
print(f"Public URL: {tunnel.public_url}")
print("Server is running in the background. Copy this URL into your local client script.")

INFO:     Started server process [91]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: https://unshakable-yasuko-luxuriantly.ngrok-free.dev                                    
Server is running in the background. Copy this URL into your local client script.


## 7. Local client (reference only — run this on YOUR machine, not in Kaggle)

```python
import requests

NGROK_URL = "https://xxxx.ngrok-free.app"  # paste from the cell above, update each restart
API_KEY = "choose-any-secret-string-here"   # must match APP_API_KEY above

def query_model(prompt: str, max_new_tokens: int = 256, temperature: float = 0.7) -> str:
    try:
        resp = requests.post(
            f"{NGROK_URL}/generate",
            json={"prompt": prompt, "max_new_tokens": max_new_tokens, "temperature": temperature},
            headers={"x-api-key": API_KEY},
            timeout=120,
        )
        resp.raise_for_status()
        return resp.json()["response"]
    except requests.exceptions.RequestException as e:
        return f"Request failed: {e}"

if __name__ == "__main__":
    print(query_model("Explain quantum entanglement simply."))
```

**Notes:**
- Every time this notebook restarts, ngrok assigns a new random URL — update `NGROK_URL` locally each time.
- This server runs against your weekly 30-hour GPU quota for as long as it stays up.
- Concurrent requests are handled sequentially (no request queue/batching) — fine for personal/dev use, not for multi-user load.